# AML music project notebook
This notebook runs your MTAT-first pipeline end to end while keeping model code in `scripts/` and `src/aml_music/`.
It is a thin orchestration layer: each section triggers the same CLI steps you already use in the terminal.

Run all blocks for a full experiment pass, or run only the branch you are currently debugging.


## 0) setup
This block finds the repository root, sets shared paths, and locks execution to the notebook's Python environment.
It also defines `run_cmd(...)`, which streams command output live and fails fast on non-zero exit codes.

That gives you one consistent runtime context for all later steps.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "scripts").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find project root. Open this notebook inside the project repo.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)

PYTHON = sys.executable

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MANIFEST_DIR = ARTIFACTS_DIR / "manifests"
RUNS_DIR = ARTIFACTS_DIR / "runs"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

MTAT_ROOT = PROJECT_ROOT / "MTAT"
GTZAN_ROOT = PROJECT_ROOT / "GTZAN"
ANNOTATIONS_PATH = MTAT_ROOT / "annotations_final.csv"

MTAT_MANIFEST = MANIFEST_DIR / "mtat_top50_manifest.csv"
MTAT_TAGS = MANIFEST_DIR / "mtat_top50_tags.json"

MTAT_BASELINE_DIR = RUNS_DIR / "mtat_cnn_baseline"
MTAT_BASELINE_CKPT = MTAT_BASELINE_DIR / "best.pt"
MTAT_EVAL_DIR = REPORTS_DIR / "mtat_eval"


def run_cmd(args: list[str]) -> None:
    cmd = [str(x) for x in args]
    print("$", " ".join(shlex.quote(x) for x in cmd))
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if process.stdout is None:
        raise RuntimeError("Failed to capture process output.")
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")


print("Project root:", PROJECT_ROOT)
print("Python:", PYTHON)


## 1) quick setup check
This verifies that MTAT and GTZAN folders exist and reports file counts before you start long jobs.
It also checks whether `annotations_final.csv` is present, which is required to build MTAT labels and splits.

Use this as a quick sanity gate before training.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/check_setup.py",
    "--mtat-root", str(MTAT_ROOT),
    "--gtzan-root", str(GTZAN_ROOT),
    "--annotations", str(ANNOTATIONS_PATH),
])


## 2) download MTAT annotations if missing
Run this only when `MTAT/annotations_final.csv` is missing.
The cell calls the project downloader script and writes the annotation CSV into the expected MTAT path.

If the file already exists, the cell skips download to avoid unnecessary network calls.


In [ ]:
if ANNOTATIONS_PATH.exists():
    print(f"Already present: {ANNOTATIONS_PATH}")
else:
    run_cmd([
        PYTHON,
        "scripts/download_mtat_annotations.py",
        "--output", str(ANNOTATIONS_PATH),
    ])


## 3) build manifests
This creates the main MTAT Top-50 manifest and the auxiliary GTZAN manifest.
The MTAT step also writes split-audit artifacts (clip counts, track counts, overlap checks) so you can confirm split hygiene.

Core outputs land in `artifacts/manifests/` and are reused by all training/evaluation scripts.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/build_mtat_manifest.py",
    "--mtat-root", str(MTAT_ROOT),
    "--annotations", str(ANNOTATIONS_PATH),
])

run_cmd([
    PYTHON,
    "scripts/build_gtzan_manifest.py",
    "--gtzan-root", str(GTZAN_ROOT),
])


## 3b) direct backend call example
This is a pure function-level example, not a CLI call.
It imports `load_audio`, `chunk_audio`, and `LogMelFrontend` directly from `src/aml_music` to show how the backend pieces connect.

The printed shapes help confirm that decode, chunking, and frontend extraction work in your notebook kernel.


In [ ]:
import pandas as pd

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from aml_music.audio import chunk_audio, load_audio
from aml_music.features.logmel import LogMelFrontend

if not MTAT_MANIFEST.exists():
    raise RuntimeError(f"Missing manifest: {MTAT_MANIFEST}. Run the previous cell first.")

frame = pd.read_csv(MTAT_MANIFEST).head(1)
audio_path = frame.iloc[0]["audio_path"]
audio, _ = load_audio(audio_path, sample_rate=16000)
chunks = chunk_audio(audio, chunk_size=16000 * 3, hop_size=16000 * 1)
frontend = LogMelFrontend(sample_rate=16000)
mel = frontend(chunks[0])
print("audio_path:", audio_path)
print("num_chunks:", len(chunks))
print("first_mel_shape:", mel.shape)


## 4) extract log-mel features (optional cache)
This precomputes log-mel arrays to `artifacts/features/logmel`.
Training does not require this cache, but caching is useful when you rerun many experiments with the same frontend setup.

With `--on-error skip`, damaged files are logged and the rest of the extraction continues.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/extract_logmel.py",
    "--manifest", str(MTAT_MANIFEST),
    "--on-error", "skip",
])


## 5) train MTAT short-chunk CNN baseline
This trains the primary baseline: short-chunk CNN on MTAT with log-mel input.
The script handles batching, validation, early stopping, and saves `best.pt` plus training history.

This checkpoint is the anchor model for later evaluation and transfer blocks.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/train_mtat_cnn.py",
    "--manifest", str(MTAT_MANIFEST),
    "--tags", str(MTAT_TAGS),
    "--output-dir", str(MTAT_BASELINE_DIR),
    "--on-bad-audio", "skip",
])


## 6) evaluate baseline
This runs full MTAT evaluation from the saved checkpoint.
It includes pooling comparisons, duration sweeps, rare-tag analysis, and robustness tests under simple perturbations.

Summary and detailed metrics are written to `artifacts/reports/mtat_eval`.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/evaluate_mtat.py",
    "--manifest", str(MTAT_MANIFEST),
    "--tags", str(MTAT_TAGS),
    "--checkpoint", str(MTAT_BASELINE_CKPT),
    "--output-dir", str(MTAT_EVAL_DIR),
])


## 7) optional: saliency maps
This generates qualitative maps for quick inspection of model focus regions.
It backpropagates from the top predicted tag and saves spectrogram + saliency visual pairs.

Use this for error analysis and report examples, not as a primary metric.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/export_saliency.py",
    "--manifest", str(MTAT_MANIFEST),
    "--tags", str(MTAT_TAGS),
    "--checkpoint", str(MTAT_BASELINE_CKPT),
])


## 8) optional: waveform ablation
This trains the raw-waveform branch under the same MTAT task setup.
Keeping protocol similar to the log-mel baseline makes the representation comparison cleaner.

Outputs go to `artifacts/runs/mtat_waveform` for side-by-side result checks.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/train_mtat_waveform.py",
    "--manifest", str(MTAT_MANIFEST),
    "--tags", str(MTAT_TAGS),
    "--output-dir", str(RUNS_DIR / "mtat_waveform"),
    "--on-bad-audio", "skip",
])


## 9) optional: pretrained branch
This runs the pretrained-encoder experiment (`panns` or `musicnn`) with a lightweight downstream head.
On Python 3.13, `musicnn` may be unavailable; in non-strict mode the script falls back to `panns` and records that in `summary.json`.

Use this block to benchmark transfer-ready embeddings against from-scratch training.


In [ ]:
PRETRAIN_BACKEND = "panns"  # "panns" or "musicnn"

run_cmd([
    PYTHON,
    "scripts/train_mtat_pretrained.py",
    "--manifest", str(MTAT_MANIFEST),
    "--tags", str(MTAT_TAGS),
    "--backend", PRETRAIN_BACKEND,
    "--output-dir", str(RUNS_DIR / "mtat_pretrained"),
    "--on-bad-audio", "skip",
])


## 10) optional: GTZAN transfer probe
This evaluates representation transfer from MTAT to GTZAN.
It extracts embeddings with a frozen MTAT encoder and compares them against a random-encoder baseline using the same linear classifier protocol.

The script also supports bad-audio handling and writes failure logs when unreadable files are encountered.


In [ ]:
run_cmd([
    PYTHON,
    "scripts/train_transfer_gtzan.py",
    "--gtzan-root", str(GTZAN_ROOT),
    "--mtat-checkpoint", str(MTAT_BASELINE_CKPT),
    "--output-dir", str(RUNS_DIR / "gtzan_transfer"),
    "--on-bad-audio", "skip",
])


## 11) read summaries
This is a compact result viewer for the main JSON outputs.
It prints available summaries for baseline, evaluation, ablations, pretrained branch, and GTZAN transfer.

Use it to verify run health before deeper analysis in separate plotting/report notebooks.


In [2]:
def show_json(path: Path) -> None:
    if not path.exists():
        print(f"Missing: {path}")
        return
    print(f"=== {path} ===")
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, indent=2))


    show_json(MTAT_BASELINE_DIR / "summary.json")
    show_json(MTAT_EVAL_DIR / "summary.json")
    show_json(RUNS_DIR / "mtat_waveform" / "summary.json")
    show_json(RUNS_DIR / "mtat_pretrained" / "summary.json")
    show_json(RUNS_DIR / "gtzan_transfer" / "summary.json")
